# Synopsys Careers Scraper — Armenia Jobs

**Source:** [careers.synopsys.com](https://careers.synopsys.com/search-jobs)  
**Company:** Synopsys — EDA software and semiconductor IP leader  
**ATS:** Avature (server-rendered HTML + JSON-LD on detail pages)  
**Purpose:** Collect all Yerevan/Armenia IT job postings for thesis NLP analysis.

### Strategy
1. Fetch listing page filtered to Yerevan: `/search-jobs?location=Yerevan`
2. Collect all `/job/yerevan/` links from HTML
3. For each detail page: extract `<script type="application/ld+json">` JobPosting block
4. Strip HTML from `description` field → `full_text`
5. Save raw + standardized CSVs

### Ethics & robots.txt
- Synopsys careers site is publicly accessible (no login required)
- `robots.txt` allows crawling of `/search-jobs` and `/job/` paths
- Rate-limited: 1.5 s between requests
- User-Agent identifies academic research purpose

In [1]:
import json
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path
from datetime import date

BASE_URL  = "https://careers.synopsys.com"
HEADERS   = {
    "User-Agent": "Mozilla/5.0 (compatible; ThesisResearch/1.0; Armenian IT curriculum alignment; academic use)",
    "Accept-Language": "en-US,en;q=0.9",
}
DELAY_S   = 1.5
TODAY     = date.today().isoformat()

BASE_DIR  = Path.cwd()
RAW_DIR   = BASE_DIR / "data" / "raw" / "jobs"
PROC_DIR  = BASE_DIR / "data" / "processed" / "jobs"

print(f"Today : {TODAY}")
print(f"Raw   : {RAW_DIR}")
print(f"Proc  : {PROC_DIR}")

Today : 2026-03-22
Raw   : /Users/lianaaghamalyan/thesis_data/data/raw/jobs
Proc  : /Users/lianaaghamalyan/thesis_data/data/processed/jobs


## Helper functions

In [1]:
def html_to_text(html: str) -> str:
    """Strip HTML tags and normalize whitespace."""
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    # Replace block elements with newlines for readability
    for tag in soup.find_all(["p", "li", "br", "h1", "h2", "h3", "h4"]):
        tag.insert_before("\n")
    text = soup.get_text(" ")
    # Collapse whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def get_jsonld_job(soup) -> dict:
    """Extract the first application/ld+json block with @type JobPosting."""
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string)
            # Handle single object or array
            if isinstance(data, list):
                for item in data:
                    if item.get("@type") == "JobPosting":
                        return item
            elif data.get("@type") == "JobPosting":
                return data
        except (json.JSONDecodeError, AttributeError):
            continue
    return {}


def normalize_date(val: str) -> str:
    """Normalize date strings to YYYY-MM-DD."""
    if not val:
        return ""
    val = val.strip()
    # Already ISO: 2026-03-20 or 2026-3-20
    m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})", val)
    if m:
        return f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    # M/D/YYYY
    m = re.match(r"^(\d{1,2})/(\d{1,2})/(\d{4})$", val)
    if m:
        return f"{m.group(3)}-{int(m.group(1)):02d}-{int(m.group(2)):02d}"
    return val


print("Helpers defined.")

Helpers defined.


## Step 1 — Collect Yerevan job URLs from listing page

In [1]:
SEARCH_URL = f"{BASE_URL}/search-jobs?location=Yerevan"

resp = requests.get(SEARCH_URL, headers=HEADERS, timeout=20)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

# Job cards are <a> tags whose href matches /job/yerevan/
job_links = []
seen = set()
for a in soup.find_all("a", href=True):
    href = a["href"]
    if "/job/yerevan/" in href.lower() or "/job/armenia/" in href.lower():
        full_url = BASE_URL + href if href.startswith("/") else href
        if full_url not in seen:
            seen.add(full_url)
            job_links.append(full_url)

print(f"Search URL : {SEARCH_URL}")
print(f"HTTP status: {resp.status_code}")
print(f"Job links found: {len(job_links)}")
print()
for url in job_links:
    print(f"  {url}")

Search URL : https://careers.synopsys.com/search-jobs?location=Yerevan
HTTP status: 200
Job links found: 2

  https://careers.synopsys.com/job/yerevan/software-engineering-intern/44408/93005893984
  https://careers.synopsys.com/job/yerevan/product-publications-intern/44408/93005734224


## Step 2 — Scrape detail pages

For each job URL: extract JSON-LD `JobPosting` block (title, company, location, datePosted, validThrough, description).  
Strip HTML from `description` → `full_text`.

In [1]:
records = []
errors = 0

for i, url in enumerate(job_links, 1):
    print(f"[{i}/{len(job_links)}] {url}")
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
    except Exception as e:
        print(f"  ERROR: {e}")
        errors += 1
        time.sleep(DELAY_S)
        continue

    detail_soup = BeautifulSoup(r.text, "html.parser")
    jld = get_jsonld_job(detail_soup)

    if not jld:
        print("  WARNING: No JSON-LD JobPosting found, falling back to HTML")

    # Core fields from JSON-LD
    job_title    = jld.get("title", "")
    company_name = "Synopsys"
    location_raw = jld.get("jobLocation", "")
    if isinstance(location_raw, dict):
        addr = location_raw.get("address", {})
        location = f"{addr.get('addressLocality', '')}, {addr.get('addressCountry', 'Armenia')}".strip(", ")
    elif isinstance(location_raw, str):
        location = location_raw
    else:
        location = "Yerevan, Armenia"

    posting_date = normalize_date(jld.get("datePosted", ""))
    deadline     = normalize_date(jld.get("validThrough", ""))
    description_html = jld.get("description", "")
    full_text    = html_to_text(description_html)

    # Employment type and seniority from page HTML (not always in JSON-LD)
    page_text = detail_soup.get_text(" ")
    employment_type = ""
    hire_type_m = re.search(r"(?:Hire Type|Employment Type)[:\s]+([A-Za-z/ -]+)", page_text)
    if hire_type_m:
        employment_type = hire_type_m.group(1).strip()
    if not employment_type:
        employment_type = jld.get("employmentType", "")

    # Seniority from title keywords
    title_lower = job_title.lower()
    if "intern" in title_lower:
        seniority_level = "Internship"
    elif "senior" in title_lower or "staff" in title_lower or "principal" in title_lower:
        seniority_level = "Senior"
    elif "junior" in title_lower:
        seniority_level = "Junior"
    else:
        seniority_level = ""

    # Industry
    industry = "Electronic Design Automation; Semiconductor"

    records.append({
        "source":           "synopsys",
        "source_url":       url,
        "job_title":        job_title,
        "company_name":     company_name,
        "location":         location,
        "employment_type":  employment_type,
        "seniority_level":  seniority_level,
        "industries":       industry,
        "posting_date":     posting_date,
        "deadline":         deadline,
        "skills_tags":      "",
        "full_text":        full_text,
    })

    print(f"  ✓  {job_title} | posted {posting_date} | full_text {len(full_text)} chars")
    time.sleep(DELAY_S)

print(f"\nScraped: {len(records)} | Errors: {errors}")

[1/2] https://careers.synopsys.com/job/yerevan/software-engineering-intern/44408/93005893984
  ✓  Software Engineering Intern | posted 2026-03-20 | full_text 4812 chars
[2/2] https://careers.synopsys.com/job/yerevan/product-publications-intern/44408/93005734224
  ✓  Product Publications Intern | posted 2026-03-19 | full_text 4237 chars

Scraped: 2 | Errors: 0


## Step 3 — Save raw and standardized CSVs

In [1]:
df = pd.DataFrame(records)

# Raw
raw_path = RAW_DIR / "synopsys_jobs_raw.csv"
df.to_csv(raw_path, index=False, encoding="utf-8")
print(f"Raw saved : {raw_path}  ({len(df)} rows)")

# Standardized — canonical schema
STD_COLS = [
    "source", "source_url", "job_title", "company_name", "location",
    "employment_type", "seniority_level", "industries",
    "posting_date", "deadline", "skills_tags", "full_text",
]
std_df = df[STD_COLS]
std_path = PROC_DIR / "synopsys_jobs_standardized.csv"
std_df.to_csv(std_path, index=False, encoding="utf-8")
print(f"Standardized saved: {std_path}  ({len(std_df)} rows)")

Raw saved : /Users/lianaaghamalyan/thesis_data/data/raw/jobs/synopsys_jobs_raw.csv  (2 rows)
Standardized saved: /Users/lianaaghamalyan/thesis_data/data/processed/jobs/synopsys_jobs_standardized.csv  (2 rows)


In [1]:
print("=== FIELD COVERAGE ===")
for col in STD_COLS:
    filled = std_df[col].replace("", pd.NA).notna().sum()
    pct = filled / len(std_df) * 100
    print(f"  {col:20s}: {filled}/{len(std_df)} ({pct:.0f}%)")

print()
ft = std_df["full_text"].str.len()
print(f"full_text — Min: {ft.min()}  Median: {ft.median():.0f}  Max: {ft.max()}")
print()
print("=== JOBS SCRAPED ===")
for _, row in std_df.iterrows():
    print(f"  {row['job_title']} @ {row['company_name']}  [{row['seniority_level']}]  posted={row['posting_date']}")

=== FIELD COVERAGE ===
  source              : 2/2 (100%)
  source_url          : 2/2 (100%)
  job_title           : 2/2 (100%)
  company_name        : 2/2 (100%)
  location            : 2/2 (100%)
  employment_type     : 2/2 (100%)
  seniority_level     : 2/2 (100%)
  industries          : 2/2 (100%)
  posting_date        : 2/2 (100%)
  deadline            : 2/2 (100%)
  skills_tags         : 0/2 (0%)
  full_text           : 2/2 (100%)

full_text — Min: 4237  Median: 4524  Max: 4812

=== JOBS SCRAPED ===
  Software Engineering Intern @ Synopsys  [Internship]  posted=2026-03-20
  Product Publications Intern @ Synopsys  [Internship]  posted=2026-03-19
